# LogiSense — 05 | Advanced Data Mining: Clustering, Association Rules & Anomaly Detection

### Objective
Use unsupervised mining to discover customer groups, operational patterns and unusual orders.

### DWM topics
**K-Means · hierarchical-style comparison through silhouette/inertia · association rules · Apriori · support · confidence · lift · anomaly detection · profiling · business interpretation**

In [1]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.ensemble import IsolationForest

try:
    from mlxtend.preprocessing import TransactionEncoder
    from mlxtend.frequent_patterns import apriori, association_rules
    MLXTEND_OK=True
except Exception:
    MLXTEND_OK=False
    print("mlxtend is not installed. Install it with: pip install mlxtend")

def find_dataset():
    p=Path("../data/raw/E-commerce_Delivery_Shipping_Data_2026.csv")
    if p.exists(): return p
    csvs=list(Path("../data/raw").glob("*.csv"))
    if len(csvs)==1: return csvs[0]
    raise FileNotFoundError("Put raw CSV in data/raw/")
df=pd.read_csv(find_dataset())
print(df.shape)

(50000, 30)


## 1. Customer-level feature engineering

Clustering individual order rows is usually less meaningful than clustering customers by behavior.

We create:
- order frequency,
- total/average spend,
- average delay,
- average rating,
- return rate,
- average shipping cost.

In [2]:
if "customer_id" in df.columns:
    customer=df.groupby("customer_id").agg(
        order_count=("order_id","count"),
        total_spend=("order_value_usd","sum"),
        avg_order_value=("order_value_usd","mean"),
        avg_delay=("delivery_delay_days","mean"),
        avg_rating=("customer_rating","mean"),
        return_rate=("return_requested",lambda s:s.astype(str).str.lower().eq("yes").mean()),
        avg_shipping_cost=("shipping_cost_usd","mean")
    ).reset_index()

    features=["order_count","total_spend","avg_order_value","avg_delay","avg_rating","return_rate","avg_shipping_cost"]
    customer[features]=customer[features].replace([np.inf,-np.inf],np.nan).fillna(0)
    display(customer.head())

,customer_id,order_count,total_spend,avg_order_value,avg_delay,avg_rating,return_rate,avg_shipping_cost
0,CUS-000001,7,1065.83,152.261429,0.428571,3.714286,0.142857,121.598571
1,CUS-000002,4,325.98,81.495000,1.500000,3.000000,0.250000,156.992500
2,CUS-000003,13,2188.97,168.382308,1.307692,3.307692,0.230769,123.836923
3,CUS-000004,4,719.85,179.962500,2.000000,2.750000,0.250000,57.777500
4,CUS-000005,9,2127.99,236.443333,1.333333,3.444444,0.222222,72.713333


## 2. Choose K using inertia + silhouette

**Inertia** rewards compact clusters.  
**Silhouette** balances within-cluster compactness and between-cluster separation.

We inspect both instead of choosing K arbitrarily.

In [3]:
if "customer" in globals():
    X=StandardScaler().fit_transform(customer[features])
    rows=[]
    for k in range(2,9):
        km=KMeans(n_clusters=k,n_init=15,random_state=42)
        lab=km.fit_predict(X)
        rows.append([k,km.inertia_,silhouette_score(X,lab)])
    k_table=pd.DataFrame(rows,columns=["k","inertia","silhouette"])
    display(k_table.round(4))
    best_k=int(k_table.loc[k_table.silhouette.idxmax(),"k"])
    print("Selected K by maximum silhouette:",best_k)

,k,inertia,silhouette
0,2,36411.0954,0.1885
1,3,30786.4311,0.1770
2,4,28081.6997,0.1530
3,5,26007.6778,0.1556
4,6,24639.0269,0.1347
5,7,23433.2856,0.1386
6,8,22359.9034,0.1398


Selected K by maximum silhouette: 2


## 3. Compare K-Means with Agglomerative Clustering

This is an important comparison: two different clustering philosophies can produce different customer segments.

In [4]:
if "customer" in globals():
    compare=[]
    for k in range(2,7):
        km=KMeans(n_clusters=k,n_init=15,random_state=42).fit(X)
        ac=AgglomerativeClustering(n_clusters=k).fit(X)
        compare.append([k,"KMeans",silhouette_score(X,km.labels_)])
        compare.append([k,"Agglomerative",silhouette_score(X,ac.labels_)])
    cluster_compare=pd.DataFrame(compare,columns=["k","algorithm","silhouette"])
    display(cluster_compare.round(4))

,k,algorithm,silhouette
0,2,KMeans,0.1885
1,2,Agglomerative,0.1694
2,3,KMeans,0.1770
3,3,Agglomerative,0.1195
4,4,KMeans,0.1530
5,4,Agglomerative,0.0814
6,5,KMeans,0.1556
7,5,Agglomerative,0.0806
8,6,KMeans,0.1347
9,6,Agglomerative,0.0707


## 4. Final customer segments

A cluster number means nothing by itself. We profile each segment and assign a business interpretation.

In [5]:
if "customer" in globals():
    km=KMeans(n_clusters=best_k,n_init=15,random_state=42)
    customer["cluster"]=km.fit_predict(X)
    profile=customer.groupby("cluster")[features].mean()
    profile["customers"]=customer.groupby("cluster").size()
    display(profile.sort_values("customers",ascending=False).round(2))

,order_count,total_spend,avg_order_value,avg_delay,avg_rating,return_rate,avg_shipping_cost,customers
cluster,,,,,,,,
1,7.07,845.53,122.34,1.25,3.39,0.17,118.19,4172
0,9.42,2467.05,284.28,1.33,3.35,0.17,124.80,2176


## 5. Association Rule Mining

We convert order attributes into transaction-style items.

Examples:
`product=Electronics`, `shipping=Express`, `priority=High`, `weather=Rain`

We report:

- **Support:** how frequently a combination occurs.
- **Confidence:** how often the consequent occurs when the antecedent occurs.
- **Lift:** how much more often the combination occurs than expected by chance.

Lift > 1 suggests positive association, but does **not** prove causation.

In [6]:
if MLXTEND_OK:
    attrs=[c for c in ["product_category","shipping_method","package_size","order_priority","weather_condition","carrier"] if c in df.columns]
    transactions=[]
    for _,row in df[attrs].dropna().iterrows():
        transactions.append([f"{c}={row[c]}" for c in attrs])

    te=TransactionEncoder()
    basket=pd.DataFrame(te.fit(transactions).transform(transactions),columns=te.columns_)
    freq=apriori(basket,min_support=max(0.01,2/len(basket)),use_colnames=True,max_len=3)
    rules=association_rules(freq,metric="lift",min_threshold=1.05)
    rules=rules.sort_values(["lift","confidence","support"],ascending=False)

    display(freq.sort_values("support",ascending=False).head(20))
    display(rules[["antecedents","consequents","support","confidence","lift"]].head(25))

,support,itemsets
10,0.44900,frozenset({order_priority=Normal})
13,0.37296,frozenset({package_size=Medium})
32,0.34054,frozenset({shipping_method=Standard})
30,0.31678,frozenset({shipping_method=International})
8,0.30350,frozenset({order_priority=High})
33,0.29864,frozenset({weather_condition=Clear})
15,0.29120,frozenset({package_size=Small})
12,0.27230,frozenset({package_size=Large})
34,0.25164,frozenset({weather_condition=Cloudy})
29,0.19392,frozenset({shipping_method=Express})


,antecedents,consequents,support,confidence,lift
302,"frozenset({package_size=Oversized, order_prior...",frozenset({product_category=Automotive}),0.01174,0.430037,7.394027
305,frozenset({product_category=Automotive}),"frozenset({package_size=Oversized, order_prior...",0.01174,0.201857,7.394027
303,"frozenset({order_priority=Normal, product_cate...",frozenset({package_size=Oversized}),0.01174,0.468476,7.372925
304,frozenset({package_size=Oversized}),"frozenset({order_priority=Normal, product_cate...",0.01174,0.184766,7.372925
43,frozenset({product_category=Automotive}),frozenset({package_size=Oversized}),0.02716,0.466988,7.349506
42,frozenset({package_size=Oversized}),frozenset({product_category=Automotive}),0.02716,0.427447,7.349506
364,frozenset({product_category=Grocery}),"frozenset({weather_condition=Cloudy, package_s...",0.01186,0.146275,2.129189
363,"frozenset({weather_condition=Cloudy, package_s...",frozenset({product_category=Grocery}),0.01186,0.172635,2.129189
359,"frozenset({weather_condition=Clear, package_si...",frozenset({product_category=Grocery}),0.01384,0.172354,2.125724
360,frozenset({product_category=Grocery}),"frozenset({weather_condition=Clear, package_si...",0.01384,0.170696,2.125724


## 6. Association-rule comparison by support threshold

A rule can look impressive at very low support but affect almost no orders.  
We therefore compare thresholds and inspect how many actionable rules remain.

In [7]:
if MLXTEND_OK:
    rows=[]
    for sup in [.01,.02,.03,.05,.08]:
        fr=apriori(basket,min_support=sup,use_colnames=True,max_len=3)
        if len(fr):
            rr=association_rules(fr,metric="lift",min_threshold=1.05)
            strong=rr[rr["lift"]>1.2]
            rows.append([sup,len(fr),len(rr),len(strong),
                         strong["lift"].mean() if len(strong) else np.nan])
    display(pd.DataFrame(rows,columns=["min_support","frequent_itemsets","rules","lift>1.2","avg_lift_strong"]).round(3))

,min_support,frequent_itemsets,rules,lift>1.2,avg_lift_strong
0,0.01,956,498,310,1.853
1,0.02,382,98,84,1.887
2,0.03,245,36,34,1.741
3,0.05,98,6,6,1.750
4,0.08,58,2,2,1.799


## 7. Anomaly detection

Isolation Forest identifies unusual operational orders without requiring labels.

Potential uses:
- unusually expensive freight,
- extreme distance,
- unusual processing time,
- abnormal delivery attempts,
- suspicious combinations of operational values.

An anomaly is a **review signal**, not proof of fraud or error.

In [8]:
anom_cols=[c for c in [
    "product_weight_kg","order_value_usd","distance_km","promised_delivery_days",
    "actual_delivery_days","delivery_delay_days","shipping_cost_usd",
    "delivery_attempts","warehouse_processing_hours"
] if c in df.columns]
if len(anom_cols)>=3:
    Z=df[anom_cols].apply(pd.to_numeric,errors="coerce").fillna(df[anom_cols].median())
    iso=IsolationForest(n_estimators=250,contamination=.02,random_state=42)
    df["anomaly_flag"]=iso.fit_predict(StandardScaler().fit_transform(Z))
    df["anomaly_score"]=iso.decision_function(StandardScaler().fit_transform(Z))
    print("Anomaly count:",(df.anomaly_flag==-1).sum())
    display(df.loc[df.anomaly_flag==-1,anom_cols+["anomaly_score"]].sort_values("anomaly_score").head(20))

Anomaly count: 1000


,product_weight_kg,order_value_usd,distance_km,promised_delivery_days,actual_delivery_days,delivery_delay_days,shipping_cost_usd,delivery_attempts,warehouse_processing_hours,anomaly_score
5085,4.62,353.15,7458.5,14,23,9,500.00,3,47.5,-0.142343
25320,3.96,49.86,8735.3,14,25,11,500.00,2,46.9,-0.122421
45750,23.58,250.57,3948.8,12,21,9,469.21,2,40.7,-0.122011
5698,6.43,603.04,6816.9,13,20,7,500.00,3,43.2,-0.121838
43809,1.62,169.06,7920.2,14,23,9,485.52,3,24.7,-0.121413
45009,20.45,1018.26,3423.6,12,20,8,426.21,2,28.3,-0.118836
40469,18.42,235.01,8077.5,5,11,6,500.00,2,41.8,-0.112504
28540,24.76,230.90,7780.8,14,22,8,500.00,1,5.0,-0.112313
13450,2.23,907.31,5663.4,13,21,8,451.37,3,28.8,-0.108729
19078,26.15,74.85,4655.1,12,22,10,500.00,1,44.2,-0.107411


## 8. Mining-to-business translation

| Mining output | Business use |
|---|---|
| Customer clusters | Personalization / retention / service prioritization |
| Association rules | Operational bundle patterns / policy investigation |
| Anomalies | Exception monitoring |
| High-risk clusters | Proactive delivery intervention |
| Carrier patterns | Procurement and SLA review |

This completes the **descriptive → diagnostic → predictive → mining** story of the project.